# 🏡 San Antonio Automated Apartment & Townhome Hunter (v3 - Complete Standalone Edition)
This complete notebook executes the entire end-to-end multi-criteria pipeline in place:
1. **Walmart-Excluded Grocery Proximity KD-Tree:** Skips Walmart and targets H-E-B, Trader Joe's, Sprouts, Whole Foods, and Central Market.
2. **Edwards Aquifer Water Softener / RO Detector:** Detects water softeners, softener loops, and reverse-osmosis filtration.
3. **Hard Flooring & Cat-Friendly Scorer:** Rewards 100% luxury vinyl plank / hardwood / tile and checks cat policies.
4. **Regex Junk Fee Parser:** Extracts itemized monthly corporate fees and utility inclusions.
5. **Spatial Noise KD-Trees:** Calculates distance matrices against OpenStreetMap railways and San Antonio International Airport (SAT) approach vectors.

In [ ]:
# @title Step 0: Install Required Packages (including Local CLIP Vision)
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests pandas pyyaml transformers torch pillow

In [ ]:
# @title Step 1: Configuration & Setup
import os
import re
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import Point
from scipy.spatial import cKDTree
from homeharvest import scrape_property

CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],
    'filters': {'location': 'San Antonio, TX', 'min_rent': 1200, 'max_rent': 2200, 'min_beds': 2, 'min_baths': 1.5, 'past_days': 30},
    'cutoffs': {'require_in_unit_laundry_and_dishwasher': True, 'strict_appliance_text_match': False, 'max_swri_commute_mins': 28.0, 'max_grocery_dist_m': 2400.0, 'max_park_trail_dist_m': 1600.0},
    'weights': {
        'commute_score': 0.18,
        'grocery_score': 0.12,
        'park_trail_score': 0.12,
        'pollution_highway_score': 0.08,
        'modern_interior_score': 0.12,
        'garage_bike_score': 0.08,
        'costco_score': 0.08,
        'water_softener_score': 0.06,
        'pet_flooring_score': 0.06,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.05
    }
}
print('Configuration loaded!')

In [ ]:
# @title Step 2: Build Spatial KD-Trees (Groceries, Parks, Highways, Railways)
print('Fetching OpenStreetMap geospatial layers...')
loc = CONFIG['filters']['location']

def get_osm_tree(place, tags):
    gdf = ox.features_from_place(place, tags=tags).to_crs(epsg=32614)
    xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in gdf.geometry])
    names = gdf['name'].fillna('Unnamed').tolist() if 'name' in gdf.columns else ['Unnamed']*len(xy)
    return cKDTree(xy), names

TREES = {
    'grocery': get_osm_tree(loc, {'shop': ['supermarket', 'grocery', 'greengrocer']}),
    'park': get_osm_tree(loc, {'leisure': ['park', 'nature_reserve'], 'highway': ['cycleway', 'path', 'footway']}),
    'highway': get_osm_tree(loc, {'highway': ['motorway', 'trunk']}),
    'railway': get_osm_tree(loc, {'railway': 'rail'})
}
print('All spatial KD-Trees successfully built!')

In [ ]:
# @title Step 3: Define All v3 Criteria Scoring Functions
CIRCUITY_FACTOR = 1.32
WALK_SPEED_M_MIN = 80.0
BIKE_SPEED_M_MIN = 250.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def criterion_required_appliances_gatekeeper(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    shared_only_flags = ['shared laundry', 'laundromat on site', 'no washer', 'no laundry in unit', 'coin-operated laundry']
    if any(flag in desc for flag in shared_only_flags):
        return {'appliances_status': 'REJECT: Shared/No In-Unit Laundry', 'passes_appliance_gate': False}
    in_unit_terms = ['in-unit', 'in unit', 'washer/dryer', 'washer and dryer', 'w/d included', 'w/d connections', 'laundry room', 'utility room', 'full-size washer']
    has_in_unit = any(t in desc for t in in_unit_terms)
    has_dishwasher = 'dishwasher' in desc or ('stainless' in desc and 'appliances' in desc) or (pd.notna(year_built) and year_built >= 2005)
    return {'appliances_status': 'Verified (In-Unit W/D + Dishwasher)' if (has_in_unit and has_dishwasher) else 'Likely (Verify W/D in Photos)', 'passes_appliance_gate': True}

def criterion_garage_and_bike_storage(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    prop_type = str(row.get('style', '') or row.get('type', '') or '').lower()
    parking_spaces = row.get('parking_garage', np.nan)
    score, features = 30.0, []
    if pd.notna(parking_spaces) and float(parking_spaces) >= 1:
        score += 45.0
        features.append(f'{int(float(parking_spaces))}-Car Garage')
    elif any(k in desc for k in ['attached garage', 'private garage', '1-car garage', '2-car garage', 'two-car garage', 'single car garage']):
        score += 45.0
        features.append('Private/Attached Garage')
    elif 'carport' in desc or 'covered parking' in desc:
        score += 20.0
        features.append('Covered/Carport')
    if any(k in desc for k in ['bike storage', 'bicycle storage', 'bike rack', 'storage unit', 'storage closet']):
        score += 25.0
        features.append('Dedicated Bike/Gear Storage')
    return {'garage_bike_features': ', '.join(features) if features else 'Standard Surface Parking', 'garage_bike_score': round(float(np.clip(score, 0.0, 100.0)), 1)}

def criterion_costco_proximity(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    best_name, min_road_km = '', 999.0
    for c in ctx['config']['costco_locations']:
        clat, clon = c['coords']
        road_km = haversine_km(lat, lon, clat, clon) * 1.28
        if road_km < min_road_km:
            min_road_km, best_name = road_km, c['name']
    est_drive_mins = (min_road_km / 48.0) * 60.0 + 2.0
    score = np.clip(100 - ((est_drive_mins - 7.0) / (25.0 - 7.0)) * 100, 0, 100)
    return {'nearest_costco': best_name, 'costco_drive_mins': round(est_drive_mins, 1), 'costco_score': round(float(score), 1)}

def criterion_grocery_proximity(row, ctx):
    tree, names = ctx['trees']['grocery']
    distances, indices = tree.query([row['utm_x'], row['utm_y']], k=min(5, len(names)))
    if not isinstance(indices, np.ndarray):
        indices, distances = [indices], [distances]
    best_idx, best_dist, best_name = indices[0], distances[0], names[indices[0]]
    for idx, dist in zip(indices, distances):
        if 'walmart' not in names[idx].lower():
            best_idx, best_dist, best_name = idx, dist, names[idx]
            break
    net_m = best_dist * CIRCUITY_FACTOR
    score = np.clip(100 - ((net_m - 400) / (2400 - 400)) * 100, 0, 100)
    return {'nearest_grocery': best_name, 'grocery_dist_m': round(net_m), 'grocery_walk_mins': round(net_m / WALK_SPEED_M_MIN, 1), 'grocery_bike_mins': round(net_m / BIKE_SPEED_M_MIN, 1), 'grocery_score': round(float(score), 1)}

def criterion_park_trail_proximity(row, ctx):
    tree, names = ctx['trees']['park']
    dist_euc, idx = tree.query([row['utm_x'], row['utm_y']])
    net_m = dist_euc * 1.25
    score = np.clip(100 - ((net_m - 250) / (1600 - 250)) * 100, 0, 100)
    return {'nearest_park_trail': names[idx], 'park_dist_m': round(net_m), 'park_walk_mins': round(net_m / WALK_SPEED_M_MIN, 1), 'park_trail_score': round(float(score), 1)}

def criterion_swri_rush_hour_commute(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    swri_lat, swri_lon = ctx['config']['workplace']['coords']
    road_km = haversine_km(lat, lon, swri_lat, swri_lon) * 1.30
    avg_speed = 28.0 if lon < -98.68 else (50.0 if lon > -98.58 else 40.0)
    est_mins = (road_km / avg_speed) * 60.0 + 3.0
    score = np.clip(100 - ((est_mins - 8.0) / (30.0 - 8.0)) * 100, 0, 100)
    return {'swri_rush_hour_mins': round(est_mins, 1), 'commute_score': round(float(score), 1)}

def criterion_pollution_and_highway_buffer(row, ctx):
    tree, _ = ctx['trees']['highway']
    hwy_dist_m, _ = tree.query([row['utm_x'], row['utm_y']])
    score = 15.0 if hwy_dist_m < 100 else (50.0 + ((hwy_dist_m - 100) / 150.0) * 35.0 if hwy_dist_m < 250 else np.clip(85.0 + ((hwy_dist_m - 250) / 500.0) * 15.0, 85.0, 100.0))
    return {'highway_buffer_m': round(hwy_dist_m), 'pollution_highway_score': round(float(score), 1)}

def criterion_modern_appliances_and_open_floorplan(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 50.0 + sum([18 for k in ['open floor', 'open-concept', 'open concept'] if k in desc]) + sum([15 for k in ['stainless steel', 'stainless appliances'] if k in desc])
    return {'modern_interior_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_water_softener_and_filtration(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 40.0 + sum([40 for k in ['water softener', 'softener loop'] if k in desc]) + sum([25 for k in ['reverse osmosis', 'ro system'] if k in desc])
    return {'water_softener_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_pet_friendly_and_hard_flooring(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 60.0 + (25.0 if any(t in desc for t in ['vinyl plank', 'lvp', 'hardwood', 'no carpet']) else 0.0) + (15.0 if any(t in desc for t in ['cat friendly', 'pets allowed']) else 0.0)
    return {'pet_flooring_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_advanced_junk_fee_parser(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    fees = sum([int(m) for m in re.findall(r'\$(\d+)\s*(?:/mo)?\s*(?:valet|trash|waste|package|tech|fee)', desc)])
    score = np.clip(100.0 - min(fees, 50.0) + (10.0 if 'water included' in desc else 0.0), 30.0, 100.0)
    return {'extracted_monthly_fees': fees, 'effective_rent_score': round(float(score), 1)}

def criterion_spatial_noise_buffer(row, ctx):
    rail_dist = ctx['trees']['railway'][0].query([row['utm_x'], row['utm_y']])[0] if 'railway' in ctx['trees'] else 9999
    sat_dist = haversine_km(row['latitude'], row['longitude'], 29.5338, -98.4698)
    score = 90.0 - (40.0 if rail_dist < 200 else (15.0 if rail_dist < 500 else 0.0)) - (25.0 if sat_dist < 6.0 else 0.0)
    return {'noise_buffer_score': round(float(np.clip(score, 20.0, 100.0)), 1)}

ACTIVE_CRITERIA = [
    criterion_required_appliances_gatekeeper, criterion_garage_and_bike_storage,
    criterion_costco_proximity, criterion_grocery_proximity, criterion_park_trail_proximity,
    criterion_swri_rush_hour_commute, criterion_pollution_and_highway_buffer,
    criterion_modern_appliances_and_open_floorplan, criterion_water_softener_and_filtration,
    criterion_pet_friendly_and_hard_flooring, criterion_advanced_junk_fee_parser, criterion_spatial_noise_buffer
]
print('All criteria functions successfully registered!')

In [ ]:
# @title Step 5: Robust 5-Axis Kitchen Vision Grader + Visual Audit HTML Report
import io
import os
import requests
from PIL import Image
import numpy as np
import pandas as pd

def score_photos_with_local_clip(
    df_top,
    max_candidates=25,
    liked_image_urls=None,
    disliked_image_urls=None,
    export_audit_html="outputs/visual_kitchen_audit.html"
):
    """
    Robust 3-Stage Local Vision Grader (OpenAI CLIP):
    1. Strict Room Isolation: Separates actual Kitchen photos from Living Room/Flooring photos
       (so a sunlit living room can NEVER mask an outdated kitchen via `max()`).
    2. 5-Axis Pairwise Feature Decomposition (Binary Softmax per physical attribute):
       - Axis A (Appliances): Stainless steel vs. White/black plastic or exposed electric coils
       - Axis B (Countertops): Thick quartz/granite stone slab vs. Thin laminate/formica/tile
       - Axis C (Cabinetry): Modern flat-panel/shaker cabinets vs. Dated 1990s oak/honey-wood cabinets
       - Axis D (Layout): Open-concept island/bar layout vs. Cramped closed galley wall kitchen
       - Axis E (Flooring): Hardwood / Luxury Vinyl Plank (LVP) vs. Wall-to-wall carpet
    3. Optional Few-Shot Personal Calibration:
       - If you pass 1-3 URLs of kitchens you LOVE (`liked_image_urls`) and 1-3 kitchens you HATE
         (`disliked_image_urls`), it computes exact cosine similarity in CLIP embedding space
         to your personal taste!
    4. Transparency Audit HTML:
       - Outputs `outputs/visual_kitchen_audit.html` showing the exact photo selected as the
         kitchen and every sub-score so you can immediately see why a unit scored what it did.
    """
    try:
        from transformers import CLIPProcessor, CLIPModel
        import torch
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "-q", "transformers", "torch", "pillow"])
        from transformers import CLIPProcessor, CLIPModel
        import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Robust Multi-Axis CLIP Vision Engine on {device}...")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Stage 1: Strict Room Classification
    room_labels = [
        "an indoor kitchen with stove, refrigerator, sink, countertops, and cabinets", # 0: Kitchen
        "an empty indoor living room or bedroom showing floors and walls",             # 1: Living/Bedroom
        "anindoor bathroom with toilet, shower, or bathtub",                           # 2: Bathroom
        "an outdoor building exterior, swimming pool, patio, street, or grass",        # 3: Exterior
        "a 2D floor plan diagram, map, or text logo"                                   # 4: Diagram
    ]

    # Stage 2: Independent Binary Contrasts for Kitchens (evaluated ONLY on verified kitchen photos)
    kitchen_axes = {
        "appliances": [
            "modern stainless steel kitchen appliances, sleek induction or gas range, stainless refrigerator",
            "old white or black plastic kitchen appliances, exposed electric coil burners, cheap stove"
        ],
        "counters": [
            "luxury thick quartz, marble, or polished granite stone kitchen countertops with undermount sink",
            "cheap laminate formica countertops,dated beige or speckled counter, or old drop-in sink"
        ],
        "cabinets": [
            "modern shaker style or sleek flat-panel contemporary kitchen cabinets with modern hardware",
            "dated 1980s or 1990s orange oak wood cabinets, cheap builder-grade brown wood doors, or worn cabinets"
        ],
        "layout": [
            "spacious open-concept kitchen with a large center island or breakfast bar overlooking the living room",
            "cramped narrow galley kitchen, closed-off small apartment kitchen with no island and tight walls"
        ]
    }

    # Independent Binary Contrast for Flooring (evaluated on kitchen + living room photos)
    flooring_axis = [
        "clean modern luxury vinyl plank flooring, wide-plank hardwood floors, or modern large format tile",
        "wall-to-wall beige carpet, plush bedroom carpeting, or dated linoleum squares"
    ]

    def get_image_embedding(img):
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            feat = model.get_image_features(**inputs)
            feat = feat / feat.norm(p=2, dim=-1, keepdim=True)
        return feat.cpu().numpy()[0]

    def binary_axis_prob(img, pair_prompts):
        inputs = processor(text=pair_prompts, images=img, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits_per_image
            probs = logits.softmax(dim=1).cpu().numpy()[0]
        return float(probs[0]) # Probability of the positive (modern) pole vs negative pole

    # Pre-compute user reference embeddings if provided (Few-Shot Personal Taste Anchors)
    liked_embs, disliked_embs = [], []
    if liked_image_urls:
        for u in liked_image_urls:
            try:
                im = Image.open(io.BytesIO(requests.get(u, timeout=4).content)).convert("RGB")
                liked_embs.append(get_image_embedding(im))
            except Exception:
                pass
    if disliked_image_urls:
        for u in disliked_image_urls:
            try:
                im = Image.open(io.BytesIO(requests.get(u, timeout=4).content)).convert("RGB")
                disliked_embs.append(get_image_embedding(im))
            except Exception:
                pass

    scores = []
    notes = []
    audit_rows = []

    for idx, row in df_top.head(max_candidates).iterrows():
        urls = []
        if pd.notna(row.get('primary_photo')) and str(row.get('primary_photo')).startswith('http'):
            urls.append(str(row['primary_photo']))
        if pd.notna(row.get('alt_photos')):
            alt = [u.strip() for u in str(row['alt_photos']).split(',') if u.strip().startswith('http')]
            urls.extend(alt[:14]) # Scan up to 15 photos so we don't miss the kitchen!

        # Deduplicate while preserving order
        seen = set()
        urls = [u for u in urls if not (u in seen or seen.add(u))][:15]

        if not urls:
            scores.append(45.0)
            notes.append("No photo URLs (Unverified)")
            continue

        kitchen_candidates = [] # (kitchen_prob, url, img)
        living_candidates = []  # (living_prob, url, img)

        for url in urls:
            try:
                resp = requests.get(url, timeout=3.5)
                img = Image.open(io.BytesIO(resp.content)).convert("RGB")
                inputs = processor(text=room_labels, images=img, return_tensors="pt", padding=True).to(device)
                with torch.no_grad():
                    r_probs = model(**inputs).logits_per_image.softmax(dim=1).cpu().numpy()[0]

                k_prob, l_prob, b_prob, ext_prob, diag_prob = [float(x) for x in r_probs]
                # Reject exteriors, bathrooms, and floorplan diagrams
                if ext_prob > 0.35 or diag_prob > 0.35 or b_prob > 0.40:
                    continue
                if k_prob >= 0.38:
                    kitchen_candidates.append((k_prob, url, img))
                elif l_prob >= 0.38:
                    living_candidates.append((l_prob, url, img))
            except Exception:
                continue

        # Sort by highest confidence that the photo is genuinely a kitchen
        kitchen_candidates.sort(key=lambda x: x[0], reverse=True)
        living_candidates.sort(key=lambda x: x[0], reverse=True)

        if not kitchen_candidates:
            # DO NOT reward a listing if it hides or lacks kitchen photos!
            floor_scores = [binary_axis_prob(im, flooring_axis) * 100.0 for _, _, im in living_candidates[:2]]
            fallback_floor = np.mean(floor_scores) if floor_scores else 45.0
            penalized = round(float(np.clip(fallback_floor * 0.65, 25.0, 58.0)), 1)
            scores.append(penalized)
            notes.append(f"No Kitchen Photo Found (Capped: {penalized}/100)")
            continue

        # Evaluate the Top 2 clearest kitchen photos across all 4 kitchen axes
        top_k = kitchen_candidates[:2]
        axis_results = {k: [] for k in kitchen_axes}
        floor_vals = []
        taste_vals = []

        for k_conf, k_url, k_img in top_k:
            for axis_name, pair_prompts in kitchen_axes.items():
                axis_results[axis_name].append(binary_axis_prob(k_img, pair_prompts) * 100.0)
            floor_vals.append(binary_axis_prob(k_img, flooring_axis) * 100.0)

            if liked_embs or disliked_embs:
                emb = get_image_embedding(k_img)
                pos_sim = max([float(np.dot(emb, le)) for le in liked_embs]) if liked_embs else 0.25
                neg_sim = max([float(np.dot(emb, de)) for de in disliked_embs]) if disliked_embs else 0.25
                # Convert cosine margin (-0.15 to +0.15) into 0-100 score
                margin = np.clip((pos_sim - neg_sim) * 350.0 + 50.0, 0.0, 100.0)
                taste_vals.append(float(margin))

        for _, _, l_img in living_candidates[:2]:
            floor_vals.append(binary_axis_prob(l_img, flooring_axis) * 100.0)

        # Aggregate per-axis scores (using mean across kitchen views, never `max()` across non-kitchens)
        app_s = float(np.mean(axis_results["appliances"]))
        cnt_s = float(np.mean(axis_results["counters"]))
        cab_s = float(np.mean(axis_results["cabinets"]))
        lay_s = float(np.mean(axis_results["layout"]))
        flr_s = float(np.mean(floor_vals)) if floor_vals else 50.0

        # Bottleneck / "Weakest Link" Rule:
        # If cabinets, counters, OR appliances are severely dated (< 35/100), cap or heavily penalize!
        raw_weighted = (0.28 * app_s) + (0.26 * cnt_s) + (0.24 * cab_s) + (0.12 * lay_s) + (0.10 * flr_s)
        weakest_kitchen_link = min(app_s, cnt_s, cab_s)
        if weakest_kitchen_link < 35.0:
            # Drag the score down toward the weakest link so old cabinets + new fridge don't get an 85
            raw_weighted = 0.50 * raw_weighted + 0.50 * weakest_kitchen_link

        if taste_vals:
            raw_weighted = 0.70 * raw_weighted + 0.30 * float(np.mean(taste_vals))

        final_v_score = round(float(np.clip(raw_weighted, 10.0, 100.0)), 1)
        scores.append(final_v_score)

        breakdown_str = (
            f"Kitchen {final_v_score}/100 "
            f"[App:{round(app_s)} | Cnt:{round(cnt_s)} | Cab:{round(cab_s)} | Open:{round(lay_s)} | Flr:{round(flr_s)}]"
        )
        notes.append(breakdown_str)

        audit_rows.append({
            "address": row.get("formatted_address", "Listing"),
            "price": row.get("list_price", ""),
            "url": row.get("property_url", "#"),
            "kitchen_img_url": top_k[0][1],
            "kitchen_conf": round(top_k[0][0] * 100),
            "final_score": final_v_score,
            "appliances": round(app_s),
            "counters": round(cnt_s),
            "cabinets": round(cab_s),
            "layout": round(lay_s),
            "flooring": round(flr_s)
        })

    # Export visual audit HTML so the user can inspect the exact kitchen photo & sub-scores
    if export_audit_html and audit_rows:
        os.makedirs(os.path.dirname(export_audit_html), exist_ok=True)
        cards_html = ""
        for r in sorted(audit_rows, key=lambda x: x["final_score"], reverse=True):
            badge_color = "#1b7e3c" if r["final_score"] >= 75 else ("#b57c00" if r["final_score"] >= 55 else "#b82e2e")
            cards_html += f"""
            <div style="border:1px solid #ddd; border-radius:8px; width:340px; overflow:hidden; background:#fff; box-shadow:0 2px 5px rgba(0,0,0,0.08);">
                <img src="{r['kitchen_img_url']}" style="width:100%; height:210px; object-fit:cover;" />
                <div style="padding:12px; font-family:sans-serif; font-size:13px;">
                    <div style="display:flex; justify-content:space-between; align-items:center;">
                        <span style="background:{badge_color}; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;">
                            Kitchen Score: {r['final_score']}/100
                        </span>
                        <b>${r['price']}/mo</b>
                    </div>
                    <div style="margin:6px 0; font-weight:bold;">{r['address']}</div>
                    <div style="color:#444; line-height:1.5;">
                        • <b>Appliances (Stainless vs Coil):</b> {r['appliances']}/100<br>
                        • <b>Counters (Stone vs Laminate):</b> {r['counters']}/100<br>
                        • <b>Cabinets (Modern vs Dated Oak):</b> {r['cabinets']}/100<br>
                        • <b>Open Island Layout:</b> {r['layout']}/100<br>
                        • <b>Hard Flooring (No Carpet):</b> {r['flooring']}/100
                    </div>
                    <div style="margin-top:8px;">
                        <a href="{r['url']}" target="_blank" style="color:#0066cc; text-decoration:none; font-weight:bold;">Open Listing →</a>
                    </div>
                </div>
            </div>
            """
        html_doc = f"""<!DOCTYPE html>
        <html><head><meta charset="utf-8"><title>Kitchen Visual Audit Report</title></head>
        <body style="background:#f5f6f8; padding:24px; font-family:sans-serif;">
            <h2>🍳 Multi-Axis Kitchen & Interior Visual Audit Report</h2>
            <p>Showing the exact photo classified as the <b>Kitchen</b> for each candidate and its 5-axis sub-scores (weakest-link penalized):</p>
            <div style="display:flex; flex-wrap:wrap; gap:16px;">{cards_html}</div>
        </body></html>"""
        with open(export_audit_html, "w") as f:
            f.write(html_doc)
        print(f"Saved Visual Kitchen Audit card grid to {export_audit_html}!")

    df_top = df_top.copy()
    df_top.loc[df_top.head(max_candidates).index, 'vision_modern_score'] = scores
    df_top.loc[df_top.head(max_candidates).index, 'vision_notes'] = notes
    return df_top


# Optional: Paste 1-3 image URLs of kitchens you LOVE or HATE to calibrate to your exact taste:
LIKED_KITCHEN_URLS = []     # e.g. ['https://...']
DISLIKED_KITCHEN_URLS = []  # e.g. ['https://...']

scored_df = score_photos_with_local_clip(
    scored_df,
    max_candidates=25,
    liked_image_urls=LIKED_KITCHEN_URLS,
    disliked_image_urls=DISLIKED_KITCHEN_URLS,
    export_audit_html='outputs/visual_kitchen_audit.html'
)

# Blend NLP text score (25%) + Strict 5-Axis Kitchen Vision score (75%)
has_v = scored_df['vision_modern_score'].notna()
scored_df.loc[has_v, 'modern_interior_score'] = (
    0.25 * scored_df.loc[has_v, 'modern_interior_score'] +
    0.75 * scored_df.loc[has_v, 'vision_modern_score']
).round(1)

weights = CONFIG['weights']
scored_df['composite_score'] = sum(scored_df.get(k, 50.0) * w for k, w in weights.items()).round(1)
scored_df = scored_df.sort_values(by='composite_score', ascending=False).reset_index(drop=True)
scored_df.to_csv('outputs/san_antonio_ranked_rentals_v3.csv', index=False)
scored_df[['composite_score', 'formatted_address', 'list_price', 'modern_interior_score', 'vision_notes', 'swri_rush_hour_mins', 'nearest_grocery']].head(15)

In [ ]:
# @title Step 5: Strategy 1 — Free Local CLIP Vision Grader & Interactive Map Export
import io
import requests
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

MAX_CLIP_CANDIDATES = 25  # Number of top spatial finalists to visually inspect with CLIP

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading local OpenAI CLIP model on {device} (100% free, zero API keys)...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

clip_labels = [
    'a modern luxury kitchen with stainless steel appliances, quartz island, and open floor plan',
    'an outdated old apartment kitchen with old white or black coil stove and cramped closed layout',
    'a modern living room with hardwood or luxury vinyl plank flooring',
    'an outdated room with old wall-to-wall carpet'
]

vision_scores = []
vision_notes = []

print(f'Analyzing interior photos for top {min(MAX_CLIP_CANDIDATES, len(scored_df))} finalists...')
for idx, row in scored_df.head(MAX_CLIP_CANDIDATES).iterrows():
    urls = []
    if pd.notna(row.get('primary_photo')) and str(row.get('primary_photo')).startswith('http'):
        urls.append(str(row['primary_photo']))
    if pd.notna(row.get('alt_photos')):
        alt = [u.strip() for u in str(row['alt_photos']).split(',') if u.strip().startswith('http')]
        urls.extend(alt[:3])  # Inspect up to 4 photos per listing

    if not urls:
        vision_scores.append(np.nan)
        vision_notes.append('No photo URLs')
        continue

    probs_list = []
    for url in urls[:4]:
        try:
            resp = requests.get(url, timeout=4)
            img = Image.open(io.BytesIO(resp.content)).convert('RGB')
            inputs = clip_processor(text=clip_labels, images=img, return_tensors='pt', padding=True).to(device)
            with torch.no_grad():
                outputs = clip_model(**inputs)
                probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]
            # probs[0] = modern kitchen, probs[2] = modern hard flooring
            probs_list.append(float(probs[0] + probs[2]))
        except Exception:
            continue

    if probs_list:
        best_prob = max(probs_list)
        v_score = round(float(np.clip(best_prob * 115.0, 20.0, 100.0)), 1)
        vision_scores.append(v_score)
        vision_notes.append(f'CLIP Score: {v_score}/100 ({len(probs_list)} photos)')
    else:
        vision_scores.append(np.nan)
        vision_notes.append('Photos unreachable')

scored_df['vision_modern_score'] = np.nan
scored_df['vision_notes'] = 'Not in topCLIP batch'
scored_df.loc[scored_df.head(MAX_CLIP_CANDIDATES).index, 'vision_modern_score'] = vision_scores
scored_df.loc[scored_df.head(MAX_CLIP_CANDIDATES).index, 'vision_notes'] = vision_notes

# Blend NLP text score (40%) + CLIP photo score (60%) where photos were analyzed
has_v = scored_df['vision_modern_score'].notna()
scored_df.loc[has_v, 'modern_interior_score'] = (
    0.40 * scored_df.loc[has_v, 'modern_interior_score'] +
    0.60 * scored_df.loc[has_v, 'vision_modern_score']
).round(1)

# Recompute final composite score and re-rank
weights = CONFIG['weights']
scored_df['composite_score'] = sum(scored_df[k] * w for k, w in weights.items()).round(1)
scored_df = scored_df.sort_values(by='composite_score', ascending=False).reset_index(drop=True)

# Save updated CSV & Interactive Folium Map
scored_df.to_csv('outputs/san_antonio_ranked_rentals_v3.csv', index=False)

m = folium.Map(location=[29.48, -98.58], zoom_start=11, tiles='CartoDB positron')
folium.Marker(
    location=CONFIG['workplace']['coords'],
    popup='<b>Southwest Research Institute (SwRI)</b>',
    icon=folium.Icon(color='red', icon='briefcase', prefix='fa')
).add_to(m)

for _, r in scored_df.head(30).iterrows():
    color = 'darkgreen' if r['composite_score'] >= 78 else ('green' if r['composite_score'] >= 65 else 'orange')
    popup_html = f"""
    <div style='width:280px; font-family:sans-serif;'>
        <h4 style='margin:0;'>Score: {r['composite_score']}/100 (${r['list_price']}/mo)</h4>
        <b>{r.get('formatted_address', 'Address')}</b><br>
        🖼️ <b>Vision:</b> {r.get('vision_notes', 'N/A')}<br>
        🚗 <b>SwRI Commute:</b> {r['swri_rush_hour_mins']} mins<br>
        🛒 <b>Grocery (No Walmart):</b> {r['grocery_walk_mins']}m walk ({r['nearest_grocery']})<br>
        📦 <b>Costco:</b> {r['costco_drive_mins']}m drive ({r['nearest_costco']})<br>
        <a href='{r.get('property_url', '#')}' target='_blank'>View Listing →</a>
    </div>
    """
    folium.Marker(
        location=[r['latitude'], r['longitude']],
        popup=folium.Popup(popup_html, max_width=320),
        icon=folium.Icon(color=color, icon='home', prefix='fa')
    ).add_to(m)

m.save('outputs/san_antonio_sweet_spots_map_v3.html')
print('Saved CLIP-scored CSV and outputs/san_antonio_sweet_spots_map_v3.html!')
scored_df[['composite_score', 'formatted_address', 'list_price', 'modern_interior_score', 'vision_notes', 'swri_rush_hour_mins', 'nearest_grocery', 'nearest_costco']].head(15)

In [ ]:
# @title Step 6 (Saved for Later — Strategy 3): Keyless `google.colab.ai` 1-Prompt Batch Auditor
# When you want to run a single batched LLM audit on your Top 15 finalists without an API key:
# Uncomment and run this cell inside Google Colab!

# from google.colab import ai
# top15 = scored_df.head(15)[['formatted_address', 'list_price', 'description']].to_dict(orient='records')
# prompt = (
#     'You are auditing 15 San Antonio rental finalists. For each listing, extract as a JSON list:\n'
#     '1. address\n2. wd_status (Machines Included vs Hookups Only vs Unclear)\n'
#     '3. hidden_fees_summary\n4. kitchen_summary\n\nListings:\n' + str(top15)
# )
# print(ai.generate_text(prompt))